In [ ]:

import time
import math
import copy
import json
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

torch.manual_seed(42)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



Using device: cuda


In [ ]:
!pip install torchinfo -q

In [ ]:

CIFAR100_MEAN = (0.5071, 0.4865, 0.4409)
CIFAR100_STD = (0.2673, 0.2564, 0.2762)

train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

DATA_ROOT = "./data"

train_set = torchvision.datasets.CIFAR100(
    root=DATA_ROOT, train=True, download=True, transform=train_transform
)
test_set = torchvision.datasets.CIFAR100(
    root=DATA_ROOT, train=False, download=True, transform=test_transform
)


BATCH_SIZE = 64

train_loader = DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

print(f"Train samples: {len(train_set)} | Test samples: {len(test_set)}")
print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")


100%|██████████| 169M/169M [40:02<00:00, 70.3kB/s]


Train samples: 50000 | Test samples: 10000
Train batches: 782 | Test batches: 157


In [ ]:
class PatchEmbedding(nn.Module):
    """"""

    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=256):
        super().__init__()
        assert img_size % patch_size == 0, "Image size must be divisible by patch size"
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(
            in_channels, embed_dim, kernel_size=patch_size, stride=patch_size
        )

    def forward(self, x):
        # x: (B, C, H, W) -> (B, embed_dim, H/P, W/P) -> (B, num_patches, embed_dim)
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

In [ ]:

# Transformer Encoder Block

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        residual = x
        x = self.norm1(x)
        attn_out, _ = self.attn(x, x, x, need_weights=False)
        x = residual + attn_out

        residual = x
        x = self.norm2(x)
        x = residual + self.mlp(x)
        return x


In [ ]:

# Full Vision Transformer

class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size=32,
        patch_size=4,
        in_channels=3,
        num_classes=100,
        embed_dim=256,
        depth=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.1,
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.pos_drop(x)

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)
        cls_out = x[:, 0]              # classification token
        return self.head(cls_out)


In [ ]:

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)



def estimate_vit_flops_analytical(img_size, patch_size, embed_dim, depth, num_heads, mlp_ratio=4.0):
    num_patches = (img_size // patch_size) ** 2
    seq_len = num_patches + 1  # +1 for CLS token

    # Patch embedding
    patch_dim = patch_size * patch_size * 3
    flops_patch_embed = num_patches * patch_dim * embed_dim * 2

    # Per-block FLOPs
    # QKV projection: 3 * seq_len * embed_dim * embed_dim * 2
    flops_qkv = 3 * seq_len * embed_dim * embed_dim * 2
    # Attention scores (QK^T) and weighted sum (attn @ V)
    flops_attn = 2 * seq_len * seq_len * embed_dim * 2
    # Output projection
    flops_out_proj = seq_len * embed_dim * embed_dim * 2
    # MLP: two linear layers, hidden_dim = mlp_ratio * embed_dim
    hidden_dim = int(embed_dim * mlp_ratio)
    flops_mlp = 2 * seq_len * embed_dim * hidden_dim * 2

    flops_per_block = flops_qkv + flops_attn + flops_out_proj + flops_mlp
    flops_blocks = flops_per_block * depth

    # Classification head
    flops_head = embed_dim * 100 * 2

    total_flops = flops_patch_embed + flops_blocks + flops_head
    return total_flops


def estimate_flops(model, input_size=(1, 3, 32, 32), vit_config=None):

    try:
        from torchinfo import summary
        stats = summary(model, input_size=input_size, verbose=0)

        return stats.total_mult_adds * 2, "torchinfo (measured)"
    except Exception:
        pass


    if vit_config is not None:
        flops = estimate_vit_flops_analytical(**vit_config)
        return flops, "analytical formula"

    return None, "unavailable"



# Training / evaluation loop

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def train_model(model, train_loader, test_loader, device, epochs=10, lr=0.001, tag=""):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": [], "epoch_time": []}

    for epoch in range(1, epochs + 1):
        start = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        epoch_time = time.time() - start

        test_loss, test_acc = evaluate(model, test_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)
        history["epoch_time"].append(epoch_time)

        print(f"[{tag}] Epoch {epoch}/{epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Test Loss: {test_loss:.4f} Acc: {test_acc:.2f}% | "
              f"Time: {epoch_time:.1f}s")

    return history


In [ ]:
EPOCHS = 10
LR = 0.001
IMG_SIZE = 32

vit_configs = [
    {"name": "ViT-A (P4, D256, L4, H4)",  "patch_size": 4, "embed_dim": 256, "depth": 4, "num_heads": 4},
    {"name": "ViT-B (P4, D512, L8, H8)",  "patch_size": 4, "embed_dim": 512, "depth": 8, "num_heads": 8},
    {"name": "ViT-C (P8, D256, L4, H4)",  "patch_size": 8, "embed_dim": 256, "depth": 4, "num_heads": 4},
    {"name": "ViT-D (P8, D512, L8, H8)",  "patch_size": 8, "embed_dim": 512, "depth": 8, "num_heads": 8},
]


results = []

for cfg in vit_configs:
    print(f"\n{'='*70}\nTraining {cfg['name']}\n{'='*70}")

    model = VisionTransformer(
        img_size=IMG_SIZE,
        patch_size=cfg["patch_size"],
        num_classes=100,
        embed_dim=cfg["embed_dim"],
        depth=cfg["depth"],
        num_heads=cfg["num_heads"],
        mlp_ratio=4.0,
    )

    num_params = count_parameters(model)

    flops, flops_method = estimate_flops(
        model,
        input_size=(1, 3, IMG_SIZE, IMG_SIZE),
        vit_config={
            "img_size": IMG_SIZE,
            "patch_size": cfg["patch_size"],
            "embed_dim": cfg["embed_dim"],
            "depth": cfg["depth"],
            "num_heads": cfg["num_heads"],
            "mlp_ratio": 4.0,
        },
    )

    history = train_model(
        model, train_loader, test_loader, device,
        epochs=EPOCHS, lr=LR, tag=cfg["name"],
    )

    results.append({
        "model": cfg["name"],
        "params": num_params,
        "flops": flops,
        "flops_method": flops_method,
        "final_train_acc": history["train_acc"][-1],
        "final_test_acc": history["test_acc"][-1],
        "avg_epoch_time_s": sum(history["epoch_time"]) / len(history["epoch_time"]),
        "history": history,
    })

    # Free GPU memory before the next config
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



Training ViT-A (P4, D256, L4, H4)
[ViT-A (P4, D256, L4, H4)] Epoch 1/10 | Train Loss: 4.0368 Acc: 7.21% | Test Loss: 3.7959 Acc: 10.94% | Time: 36.3s
[ViT-A (P4, D256, L4, H4)] Epoch 2/10 | Train Loss: 3.7578 Acc: 11.35% | Test Loss: 3.6628 Acc: 13.05% | Time: 32.6s
[ViT-A (P4, D256, L4, H4)] Epoch 3/10 | Train Loss: 3.5934 Acc: 14.08% | Test Loss: 3.4918 Acc: 15.96% | Time: 34.4s
[ViT-A (P4, D256, L4, H4)] Epoch 4/10 | Train Loss: 3.4884 Acc: 15.98% | Test Loss: 3.4022 Acc: 17.70% | Time: 34.3s
[ViT-A (P4, D256, L4, H4)] Epoch 5/10 | Train Loss: 3.3904 Acc: 17.67% | Test Loss: 3.2671 Acc: 19.76% | Time: 33.9s
[ViT-A (P4, D256, L4, H4)] Epoch 6/10 | Train Loss: 3.3006 Acc: 19.35% | Test Loss: 3.2015 Acc: 20.81% | Time: 33.8s
[ViT-A (P4, D256, L4, H4)] Epoch 7/10 | Train Loss: 3.2130 Acc: 20.77% | Test Loss: 3.0882 Acc: 23.66% | Time: 36.3s
[ViT-A (P4, D256, L4, H4)] Epoch 8/10 | Train Loss: 3.1310 Acc: 22.40% | Test Loss: 3.0268 Acc: 24.39% | Time: 34.5s
[ViT-A (P4, D256, L4, H4)] Epo

In [ ]:

import torchvision.models as models

def build_resnet18_cifar(num_classes=100):
    model = models.resnet18(weights=None)

    # Adapt stem for 32x32 inputs (standard CIFAR adaptation)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()

    # Adapt classification head for CIFAR-100
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model


print(f"\n{'='*70}\nTraining ResNet-18 (from scratch, CIFAR-adapted)\n{'='*70}")

resnet_model = build_resnet18_cifar(num_classes=100)
resnet_num_params = count_parameters(resnet_model)

resnet_flops, resnet_flops_method = estimate_flops(
    resnet_model, input_size=(1, 3, IMG_SIZE, IMG_SIZE), vit_config=None
)
if resnet_flops is None:
    print("Note: install `torchinfo` (pip install torchinfo) for a measured "
          "ResNet-18 FLOPs figure -- no analytical fallback is implemented "
          "for CNNs in this notebook.")

resnet_history = train_model(
    resnet_model, train_loader, test_loader, device,
    epochs=EPOCHS, lr=LR, tag="ResNet-18",
)

resnet_result = {
    "model": "ResNet-18 (scratch, CIFAR-adapted)",
    "params": resnet_num_params,
    "flops": resnet_flops,
    "flops_method": resnet_flops_method,
    "final_train_acc": resnet_history["train_acc"][-1],
    "final_test_acc": resnet_history["test_acc"][-1],
    "avg_epoch_time_s": sum(resnet_history["epoch_time"]) / len(resnet_history["epoch_time"]),
    "history": resnet_history,
}

results.append(resnet_result)



Training ResNet-18 (from scratch, CIFAR-adapted)
[ResNet-18] Epoch 1/10 | Train Loss: 3.6856 Acc: 12.87% | Test Loss: 3.1618 Acc: 21.88% | Time: 44.6s
[ResNet-18] Epoch 2/10 | Train Loss: 2.8445 Acc: 27.41% | Test Loss: 2.7789 Acc: 31.66% | Time: 44.0s
[ResNet-18] Epoch 3/10 | Train Loss: 2.3158 Acc: 38.06% | Test Loss: 2.2447 Acc: 41.09% | Time: 44.5s
[ResNet-18] Epoch 4/10 | Train Loss: 1.9807 Acc: 45.66% | Test Loss: 1.9470 Acc: 47.72% | Time: 44.1s
[ResNet-18] Epoch 5/10 | Train Loss: 1.7365 Acc: 51.47% | Test Loss: 1.8274 Acc: 50.50% | Time: 44.2s
[ResNet-18] Epoch 6/10 | Train Loss: 1.5495 Acc: 56.07% | Test Loss: 1.6411 Acc: 54.66% | Time: 44.4s
[ResNet-18] Epoch 7/10 | Train Loss: 1.4028 Acc: 59.61% | Test Loss: 1.5310 Acc: 56.96% | Time: 44.1s
[ResNet-18] Epoch 8/10 | Train Loss: 1.2656 Acc: 63.06% | Test Loss: 1.4863 Acc: 58.93% | Time: 44.5s
[ResNet-18] Epoch 9/10 | Train Loss: 1.1627 Acc: 65.93% | Test Loss: 1.4460 Acc: 60.32% | Time: 44.0s
[ResNet-18] Epoch 10/10 | Train 

In [ ]:
import pandas as pd

summary_rows = []
for r in results:
    summary_rows.append({
        "Model": r["model"],
        "Parameters": f"{r['params']:,}",
        "FLOPs (fwd, B=1)": f"{r['flops']/1e6:.1f}M" if r["flops"] else "N/A",
        "FLOPs method": r["flops_method"],
        "Final Train Acc (%)": f"{r['final_train_acc']:.2f}",
        "Final Test Acc (%)": f"{r['final_test_acc']:.2f}",
        "Avg Epoch Time (s)": f"{r['avg_epoch_time_s']:.1f}",
    })

summary_df = pd.DataFrame(summary_rows)
pd.set_option("display.max_colwidth", None)
print(summary_df.to_string(index=False))

summary_df.to_csv("problem1_results_summary.csv", index=False)
print("\nSaved to problem1_results_summary.csv")


                             Model Parameters FLOPs (fwd, B=1)         FLOPs method Final Train Acc (%) Final Test Acc (%) Avg Epoch Time (s)
          ViT-A (P4, D256, L4, H4)  3,214,692             5.9M torchinfo (measured)               24.96              26.94               34.5
          ViT-B (P4, D512, L8, H8) 25,330,276            36.9M torchinfo (measured)                5.11               5.41              189.8
          ViT-C (P8, D256, L4, H4)  3,239,268             5.8M torchinfo (measured)               12.36              12.54               30.6
          ViT-D (P8, D512, L8, H8) 25,379,428            36.9M torchinfo (measured)                5.09               5.57               57.2
ResNet-18 (scratch, CIFAR-adapted) 11,220,132          1111.0M torchinfo (measured)               68.33              61.46               44.3

Saved to problem1_results_summary.csv
